## Generic Functions

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window


def add_duplicate_flags_twocols(df, partition_cols1, partition_cols2, order_col):
    
    window_spec = Window.partitionBy(partition_cols1, partition_cols2).orderBy(order_col)

    return df.withColumn("row_num", F.row_number().over(window_spec)) \
                .withColumn("rank_num", F.rank().over(window_spec)) \
                 .withColumn("is_duplicate", F.col("row_num") > 1)

def add_duplicate_flags_threecols(df, partition_cols1, partition_cols2, partition_cols3, order_col):
    
    window_spec = Window.partitionBy(partition_cols1, partition_cols2, partition_cols3).orderBy(order_col)

    return df.withColumn("row_num", F.row_number().over(window_spec)) \
                .withColumn("rank_num", F.rank().over(window_spec)) \
                 .withColumn("is_duplicate", F.col("row_num") > 1)

def add_not_null_flags(df, columns):

    df = df.withColumn("is_id_null", F.col(columns).isNull())   
    return df

def add_range_check(df, colum, min_val, max_val):

    df = df.withColumn("is_out_of_range", F.when((F.col(colum) < min_val) | (F.col(colum) > max_val), 1).otherwise(0))
    return df


    

## Clean Silver Data 

### Discount Table


In [0]:
%sql
select * from zara_lakehouse.retail_bronze.discount_bronze

In [0]:
discount_df = spark.sql("select * from zara_lakehouse.retail_bronze.discount_bronze")

discount_df = add_duplicate_flags_twocols(discount_df, "product_id", "city", "discount")
discount_df = add_not_null_flags(discount_df, "product_id")
discount_df = add_range_check(discount_df, "discount", 0, 100)

display(discount_df)



In [0]:
clean_discount_silver_df = discount_df \
                    .filter(~discount_df.is_duplicate) \
                    .filter(~discount_df.is_id_null) \
                    .filter(F.col("is_out_of_range") == 0)

display(clean_discount_silver_df)

discount_total_clean = clean_discount_silver_df.count()
print(discount_total_clean)


### Products Sold Table


In [0]:
%sql
select * from zara_lakehouse.retail_bronze.products_sold_bronze

In [0]:
products_sold_df = spark.sql("select * from zara_lakehouse.retail_bronze.products_sold_bronze")

products_sold_df = products_sold_df \
            .withColumn("timestamp", F.to_timestamp("time")) \
            .withColumn("date", F.date_format(F.to_utc_timestamp("timestamp", "UTC"), "yyyy-MM-dd")) \
            .withColumn("year_month", F.date_format(F.to_utc_timestamp("timestamp", "UTC"), "yyyy-MM"))

products_sold_df = add_duplicate_flags_threecols(products_sold_df, "product_id", "city", "pieces_sold", "date")
products_sold_df = add_not_null_flags(products_sold_df, "product_id")


display(products_sold_df)

In [0]:
clean_products_sold_silver_df = products_sold_df \
                    .filter(~products_sold_df.is_duplicate) \
                    .filter(~products_sold_df.is_id_null) 
 

display(clean_products_sold_silver_df)

products_sold_total_clean = clean_discount_silver_df.count()
print(products_sold_total_clean)

### Product Information Table

## quarantine Silver Data

### Discount Table

In [0]:
quarantine_discount_silver_df = discount_df.filter(
    (F.col("is_duplicate") == True) |
    (F.col("is_id_null") == True) |
    (F.col("is_out_of_range") == 1)
)

display(quarantine_discount_silver_df)

discount_total_quarantine = quarantine_discount_silver_df.count()
print(discount_total_quarantine)

### Products Sold Table

In [0]:
quarantine_products_sold_silver_df = products_sold_df.filter(
    (F.col("is_duplicate") == True) |
    (F.col("is_id_null") == True) 
)

display(quarantine_products_sold_silver_df)

products_sold_total_quarantine = quarantine_products_sold_silver_df.count()
print(products_sold_total_quarantine)

## Data Quality Reports

### Discount Table

In [0]:
discount_summary_df = discount_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("is_duplicate") == False, 1).otherwise(0)).alias("non_duplicate_rows"),
    F.sum(F.when(F.col("is_duplicate") == True, 1).otherwise(0)).alias("duplicate_rows"),
    F.sum(F.when(F.col("is_id_null") == True, 1).otherwise(0)).alias("null_rows"),
    F.sum(F.when(F.col("is_id_null") == False, 1).otherwise(0)).alias("non_null_rows"),
    F.sum("is_out_of_range").alias("out_of_range_rows")
).withColumn(
    "ingestion_time", F.current_timestamp()
)

display (discount_summary_df)

### Products Sold Table

In [0]:
products_sold_df

products_sold_summary_df = products_sold_df.agg(
    F.count("*").alias("total_rows"),
    F.sum(F.when(F.col("is_duplicate") == False, 1).otherwise(0)).alias("non_duplicate_rows"),
    F.sum(F.when(F.col("is_duplicate") == True, 1).otherwise(0)).alias("duplicate_rows"),
    F.sum(F.when(F.col("is_id_null") == True, 1).otherwise(0)).alias("null_rows"),
    F.sum(F.when(F.col("is_id_null") == False, 1).otherwise(0)).alias("non_null_rows"),
).withColumn(
    "ingestion_time", F.current_timestamp()
)

display (products_sold_summary_df)